In [1]:
import numpy as np
import pandas as pd

# Plantilla

Es necesario ajustar las definiciones, las fuentes de los datos y posiblemente definiciones si la ENEMDU tiene una dimensión geográfica y temporal al mismo tiempo

In [2]:
data = pd.read_stata(r"Z:\harmonized\ECU\ENEMDU\data_arm\ECU_2001m12_BID.dta") # para bases de stata
# data = pd.read_stata(r"datos/ECU_2001m12_BID.dta") # para bases de stata

## Revisar los datos

- area - area
- rn - región natural
- cuidad - ciudad
- zona - zona
- sector - sector
- vivienda - vivienda
- hogar - hogar
- persona - persona
- numpers - número de personas
- edad - edad
- fexp - factor de expansión
- ingrl - ingresos

Las variables de ingreso cambian en esta encuesta y la base de datos no tiene etiquetas, según el formulario las variables del ingreso laboral monetario de la actividad principal serían
- pe58 - En su ocupación, cuánto ganó en total
- pe59a - En el mes ... retiró de su negocio o tomó de lo que produce
- pe60 - En su ocupación como ... cuánto dinero líquido recibió por concepto de salario
- pe61 - En el mes de ... cuánto le descontaron en total por las aportaciones al IESS

También existen estas variables que parecen resumir el ingreso laboral
- ingsal5
- inglabpt
- inglabst
- inglab

De acuerdo a la redacción de la pregunta en los formularios de 2001, tomaremos pe60 como la variable de ingreso laboral monetario de la actividad principal asalariada para mantener la concordancia con el resto de los años.

Hay variables dicotomicas para cada mes (ene, feb, mar, abr, may, jun, jul, ago, sep, oct, nov, dic) que dicen si estuvo o no trabajando, se puede usar estas variables y el ingreso laboral asumiendo que cuando estaba trabajando tenía ese ingreso para intentar aproximar el salario mensual y de ahí el salario trimestral, esto solo funciona así ya que no tenemos una variable que explicite el mes, en encuestas que tengan el mes o trimestre explícito esto no sería igual.

In [3]:
data[['ingsal5', 'inglabpt', 'inglabst', 'inglab']].mean()

ingsal5     161.431183
inglabpt    165.897186
inglabst    123.570547
inglab      171.910919
dtype: float64

In [4]:
data[['pe58', 'pe59a', 'pe60', 'pe61']].mean()

pe58     161.171203
pe59a     43.308552
pe60     138.093607
pe61      45.904796
dtype: float64

Filtramos solo las columnas de interés para alivar el peso en la memoria

In [5]:
data.columns

Index(['region_BID_c', 'region_c', 'pais_c', 'anio_c', 'mes_c', 'zona_c',
       'factor_ch', 'idh_ch', 'idp_ci', 'factor_ci',
       ...
       'aguamejorada_ch', 'aguamide_ch', 'bano_ch', 'banoex_ch',
       'banomejorado_ch', 'sinbano_ch', 'aguatrat_ch', 'des1_ch', 'des2_ch',
       'cpi'],
      dtype='object', length=438)

In [6]:
data = data[['area', 'rn', 'ciudad', 'zona', 'sector', 'vivienda', 'hogar',
             'persona', 'numpers', 'edad', 'fexp', 'ingrl', 'pe58', 'pe59a',
              'pe60', 'pe61', 'ingsal5', 'inglabpt', 'inglabst', 'inglab',
              'ene', 'feb', 'mar', 'abr', 'may', 'jun', 'jul', 'ago', 'sep', 
              'oct', 'nov', 'dic']]

Creamos una variable de ingreso laboral que es igual al ingreso por asalariado

In [7]:
data['ingr'] = data['pe60']

Ingreso mensual asumiendo que las personas reciben el mismo valor reportado en 'ingr' siempre que reportan estar ocupados en un mes, ahora las variables categoricas por mes tienen diferentes leyendas y no aparecen las etiquetas en la base de 2001, planteamos estas etiquetas basandonos en la continuidad más lógica desde diciembre de 2000

En diciembre de 2000 las personas reportadas como Trabajando fueron 24743, en enero de 2001 son 24725
- 1 - Desocupado
- 2 - Buscando trabajo
- 3 - Trabajando
- 0.0

In [8]:
data['ene'].value_counts()

ene
1.0    28013
3.0    24725
2.0     1108
Name: count, dtype: int64

In [9]:
data['ingr_ene'] = data.apply(lambda x: x['ingr'] if x['ene'] == 3 else None, axis=1)
data['ingr_feb'] = data.apply(lambda x: x['ingr'] if x['feb'] == 3 else None, axis=1)
data['ingr_mar'] = data.apply(lambda x: x['ingr'] if x['mar'] == 3 else None, axis=1)
data['ingr_abr'] = data.apply(lambda x: x['ingr'] if x['abr'] == 3 else None, axis=1)
data['ingr_may'] = data.apply(lambda x: x['ingr'] if x['may'] == 3 else None, axis=1)
data['ingr_jun'] = data.apply(lambda x: x['ingr'] if x['jun'] == 3 else None, axis=1)
data['ingr_jul'] = data.apply(lambda x: x['ingr'] if x['jul'] == 3 else None, axis=1)
data['ingr_ago'] = data.apply(lambda x: x['ingr'] if x['ago'] == 3 else None, axis=1)
data['ingr_sep'] = data.apply(lambda x: x['ingr'] if x['sep'] == 3 else None, axis=1)
data['ingr_oct'] = data.apply(lambda x: x['ingr'] if x['oct'] == 3 else None, axis=1)
data['ingr_nov'] = data.apply(lambda x: x['ingr'] if x['nov'] == 3 else None, axis=1)
data['ingr_dic'] = data.apply(lambda x: x['ingr'] if x['dic'] == 3 else None, axis=1)

## Deflactamos y transformamos el ingreso

Esto deja todo en dólares constantes de 2014

In [10]:
# Carga base de datos con ipc
data_externa = pd.read_excel("data_externa.xlsx", sheet_name='datos')

# filtra año de interés
datos_actual = data_externa[data_externa['Año'] == 2001]
datos_base = data_externa[data_externa['Año'] == 2014]

Diccionarios de ipc

In [11]:
ipc_dict = {fila['trimestre']: {
        'Nacional': fila['Nacional'],
        'Guayaquil': fila['Guayaquil'],
        'Quito': fila['Quito'],
        'Cuenca': fila['Cuenca']
    }
     for _, fila in datos_actual.iterrows()
     }

ipc_base_dict = {fila['trimestre']: {
        'Nacional': fila['Nacional'],
        'Guayaquil': fila['Guayaquil'],
        'Quito': fila['Quito'],
        'Cuenca': fila['Cuenca']
    }
     for _, fila in datos_base.iterrows()
     }

### Creamos identificadores para las ciudades siguiendo los códigos del INEC y para los trimestres

In [ ]:
print(data['ciudad'][0])
fac = data['ciudad'].apply(lambda x: len(str(x))).min()
fac

010150


6

In [ ]:
# Corregimos los códigos para usarlos cómo texto
data['ciudad'] = data['ciudad'].apply(str)
data['ciudad'] = data['ciudad'].apply(lambda x: '0' + x if len(x) == (fac-1) else x)

data['ciudad_2'] = data['ciudad'].apply(lambda x: x[:2])

Diccionario ciudades disponibles

In [13]:
parroquia_dict = {
    '01': 'Cuenca',
    '09': 'Guayaquil',
    '17': 'Quito'
}

data['ciudad_asignada'] = data['ciudad_2'].apply(lambda x: parroquia_dict.get(x, 'Nacional'))

In [14]:
data['ciudad_asignada'].value_counts()

ciudad_asignada
Nacional     34550
Guayaquil    13157
Quito         7647
Cuenca        5398
Name: count, dtype: int64

### Asignamos el ipc correspondiente según ciudad correspondiente

$\begin{equation}
    ingr_{USD-base-2014}^{i} = ingr_{dólares}^{i}\left( \frac{ipc^{i}_{2014}}{ipc^{i}_{año}}\right)
\end{equation}$

Desde 2000 en adelante ya no es necesario utilizar el tipo de cambio debido al cambio de moneda

In [15]:
# Función que asigna valores correspondientes
def asigna_ipc(fila, trimestre):
    return ipc_dict.get(trimestre, {}).get(fila['ciudad_asignada'], None)

def asigna_ipc_base(fila, trimestre):
    return ipc_base_dict.get(trimestre, {}).get(fila['ciudad_asignada'], None)

In [16]:
data['ipc_t1'] = data.apply(lambda fila: asigna_ipc(fila, 1), axis=1)
data['ipc_base_t1'] = data.apply(lambda fila: asigna_ipc_base(fila, 1), axis=1)

data['ipc_t2'] = data.apply(lambda fila: asigna_ipc(fila, 2), axis=1)
data['ipc_base_t2'] = data.apply(lambda fila: asigna_ipc_base(fila, 2), axis=1)

data['ipc_t3'] = data.apply(lambda fila: asigna_ipc(fila, 3), axis=1)
data['ipc_base_t3'] = data.apply(lambda fila: asigna_ipc_base(fila, 3), axis=1)

data['ipc_t4'] = data.apply(lambda fila: asigna_ipc(fila, 4), axis=1)
data['ipc_base_t4'] = data.apply(lambda fila: asigna_ipc_base(fila, 4), axis=1)

In [17]:
# Calculamos el deflactor
data['def_t1'] = (data['ipc_base_t1'] / data['ipc_t1'])
data['def_t2'] = (data['ipc_base_t2'] / data['ipc_t2'])
data['def_t3'] = (data['ipc_base_t3'] / data['ipc_t3'])
data['def_t4'] = (data['ipc_base_t4'] / data['ipc_t4'])

In [18]:
# Ingreso real por mes
data['ingr_ene_r'] = data['ingr_ene'] * data['def_t1']
data['ingr_feb_r'] = data['ingr_feb'] * data['def_t1']
data['ingr_mar_r'] = data['ingr_mar'] * data['def_t1']
data['ingr_abr_r'] = data['ingr_abr'] * data['def_t2']
data['ingr_may_r'] = data['ingr_may'] * data['def_t2']
data['ingr_jun_r'] = data['ingr_jun'] * data['def_t2']
data['ingr_jul_r'] = data['ingr_jul'] * data['def_t3']
data['ingr_ago_r'] = data['ingr_ago'] * data['def_t3']
data['ingr_sep_r'] = data['ingr_sep'] * data['def_t3']
data['ingr_oct_r'] = data['ingr_oct'] * data['def_t4']
data['ingr_nov_r'] = data['ingr_nov'] * data['def_t4']
data['ingr_dic_r'] = data['ingr_dic'] * data['def_t4']

Ingreso mensual promedio en el trimeste

In [19]:
data['ingr_t1_r'] = (data['ingr_ene_r'] + data['ingr_feb_r'] + data['ingr_mar_r'])/3
data['ingr_t2_r'] = (data['ingr_abr_r'] + data['ingr_may_r'] + data['ingr_jun_r'])/3
data['ingr_t3_r'] = (data['ingr_jul_r'] + data['ingr_ago_r'] + data['ingr_sep_r'])/3
data['ingr_t4_r'] = (data['ingr_oct_r'] + data['ingr_nov_r'] + data['ingr_dic_r'])/3

## Regiones

In [21]:
# Corregimos los códigos para usarlos cómo texto
data['ciudad'] = data['ciudad'].apply(str)
data['ciudad'] = data['ciudad'].apply(lambda x: '0' + x if len(x) == (fac-1) else x)

data['ciudad_2'] = data['ciudad'].apply(lambda x: x[:2])

In [22]:
regiones_dict = {
    'Guayas': '09',
    'Manabí': '13',
    'El Oro': '07',
    'Los Ríos': '12',
    'Pichincha': '17',
    'Azuay': '01',
    'Galápagos': '20',
    'Sierra': ['04', '10', '05', '18', '02', '06', '03', '11'],
    'Costa, Santo Domingo': ['08', '24', '23'],
    'Amazonía': ['14', '15', '16', '19', '21', '22', '90']
}

In [23]:
codigo_region = {}
for region, codes in regiones_dict.items():
    
    if isinstance(codes, list):
        for code in codes:
            codigo_region[code] = region
    
    else:
        codigo_region[codes] = region

# Mapeo de regiones
data['region'] = data['ciudad_2'].map(codigo_region)

In [24]:
data['region'].value_counts()

region
Sierra                  15138
Guayas                  13157
Pichincha                7647
Azuay                    5398
Amazonía                 4992
Manabí                   4650
El Oro                   4076
Los Ríos                 3778
Costa, Santo Domingo     1916
Name: count, dtype: int64

## Calculo ingreso de los hogares

In [25]:
columnas_idef = ['area', 'rn', 'ciudad', 'zona', 'sector', 'vivienda', 'hogar']

data['idef_hogar'] = data[columnas_idef].astype(str).agg(''.join, axis=1)
len(data['idef_hogar'].unique())

14046

In [26]:
data[['area', 'rn', 'ciudad', 'zona', 'sector', 'vivienda', 'hogar',
      'idef_hogar', 'persona', 'numpers']]

,area,rn,ciudad,zona,sector,vivienda,hogar,idef_hogar,persona,numpers
0,1,1,010150,1,4,1,1,110101501411,4,6
1,1,1,010150,1,4,1,1,110101501411,1,6
2,1,1,010150,1,4,1,1,110101501411,3,6
3,1,1,010150,1,4,1,1,110101501411,5,6
4,1,1,010150,1,4,1,1,110101501411,6,6
...,...,...,...,...,...,...,...,...,...,...
60747,2,1,050157,999,3,3,1,21050157999331,1,4
60748,2,1,050157,999,3,3,1,21050157999331,4,4
60749,2,1,050157,999,3,4,1,21050157999341,1,1
60750,2,1,050157,999,3,5,1,21050157999351,1,2


Si todos los miembros del hogar tienen NA como ingreso, mantener NA, si al menos uno tiene un ingreso sumamos para el ingreso del hogar, así evitamos subestimar el ingreso del hogar si tenemos valores perdidos

In [27]:
# Definimos una función que sume pero devuelva NA si todos son NA
def sum_with_na(series):
    if series.isna().all():
        return pd.NA
    else:
        return series.sum(skipna=True)

In [28]:
data['ingr_t1_h'] = data.groupby('idef_hogar')['ingr_t1_r'].transform(sum_with_na)
data['ingr_t2_h'] = data.groupby('idef_hogar')['ingr_t2_r'].transform(sum_with_na)
data['ingr_t3_h'] = data.groupby('idef_hogar')['ingr_t3_r'].transform(sum_with_na)
data['ingr_t4_h'] = data.groupby('idef_hogar')['ingr_t4_r'].transform(sum_with_na)

In [29]:
data[['ingr_t1_h', 'ingr_t2_h', 'ingr_t3_h', 'ingr_t4_h']].mean()

ingr_t1_h    143.681788
ingr_t2_h    148.004673
ingr_t3_h    177.172248
ingr_t4_h     176.55819
dtype: object

In [30]:
print("Ingreso medio de un hogar t4: ", data['ingr_t4_h'].mean())
print("Mediana del ingreso de un hogar t4: ", data['ingr_t4_h'].median())

Ingreso medio de un hogar t4:  176.5581899024171
Mediana del ingreso de un hogar t4:  112.68478136006945


## Sacamos edades negativas y mayores a 100 años

In [34]:
len(data)

60752

En este caso en la variable edad tenemos números y el texto 'menos de un año' así que primero transformamos todas las filas que digan 'menos de un año' a 0

In [35]:
data['edad'] = data['edad'].apply(lambda x: x if type(x) == int else 0)

In [36]:
data = data.loc[(data['edad'] >= 0) & (data['edad'] < 100)]
len(data)

60752

## Ingreso individual descontando cargas familiares

Utilizando la metodología del autor dividimos el ingreso del hogar para la escala $(A_{i}+kC_{i})^{s}$ donde $A_{i}$ es al número de adultos, $C_{i}$ es el número de niños en el hogar $i$. $k$ es el costo en recursos de cada niño y $s$ busca reflejar las restricciones

In [37]:
k = 0.4
s = 0.9

In [38]:
# Si es necesario calcular el número de niños
data['es_nino'] = data['edad'] < 10

data['ninos'] = data.groupby('idef_hogar')['es_nino'].transform('sum')

# Si es necesario calcular el número de adultos
data['es_adulto'] = data['edad'] > 10

data['adultos'] = data.groupby('idef_hogar')['es_adulto'].transform('sum')

In [39]:
data['escala'] = (data['adultos'] + k * data['ninos']) ** s

In [40]:
data['ingr_t_t1'] = data['ingr_t1_h'] / data['escala']
data['ingr_t_t2'] = data['ingr_t2_h'] / data['escala']
data['ingr_t_t3'] = data['ingr_t3_h'] / data['escala']
data['ingr_t_t4'] = data['ingr_t4_h'] / data['escala']

In [41]:
data[['ingr_t_t1', 'ingr_t_t2', 'ingr_t_t3', 'ingr_t_t4']].mean()

ingr_t_t1    36.374727
ingr_t_t2    38.207515
ingr_t_t3     45.75147
ingr_t_t4    44.989089
dtype: object

In [42]:
print("Ingreso individual descontando cargas familiares t4: ", data['ingr_t_t4'].mean())
print("Mediana del ingreso individual descontando cargas familiares t4: ", data['ingr_t_t4'].median())

Ingreso individual descontando cargas familiares t4:  44.98908909018714
Mediana del ingreso individual descontando cargas familiares t4:  25.084047638411985


## Umbrales de pobreza

Incluimos los índices de pobreza si es posible a nivel regional para luego poder utilizar de mejor forma el factor de expansión

$\begin{equation}
    umbral_{USD-base-2014}^{i} = umbral_{año}\left( \frac{ipc^{i}_{2014}}{ipc^{i}_{año}}\right)
\end{equation}$

Con $i$ el trimestre de interés y $año$ el año de interés

Diccionario de umbral

In [43]:
umbral_dict = dict(zip(datos_actual['trimestre'], datos_actual['umbral de pobreza']))
salario_dict = dict(zip(datos_actual['trimestre'], datos_actual['salario básico unificado']))
ano = 2001

## Cálculo del índice de pobreza de Foster, Greer y Thorbecke

Para calcular un ínidce de pobreza se utiliza a Foster, Greer y Thorbecke (1984), ya que satisface algunas caracterísitcas de distribución que son positivas e igual a las enunciadas por Sen, el autor usa el mismo índice.

$\begin{equation}FGT_{\alpha} = \frac{1}{N}\sum_{i=1}^{H}\left(\frac{z-y_{i}}{z}\right)^{\alpha}\end{equation}$

Donde $z$ es el umbral de pobreza, $N$ es el número de personas en la economía, $H$ es el número de pobres (personas debajo de la línea de pobreza) $y_{i}$ es el ingreso de cada individuo. Mientras mayor es el valor de $\alpha$ mayor es el peso de los individuos más pobres, mayor $FGT$ mayor pobreza en la economía.

En este caso los umbrales están anivel nacional, aún así buscamos calcular la pobreza por región y sacar un promedio ponderado por región para la pobreza nacional, con el objetivo de hacerlo más específico

## Calculo del índice de desigualdad de Atkinson

Vamos a calcular el índice de desigualdad de atkinson con un parámetro $\epsilon$ de aversión a la desigualdad y un $\mu$ que es igual a la media de los ingresos individuales, con la siguiente fórmula.

$\begin{equation}A = 1-\frac{1}{\mu}\left(\frac{1}{N}\sum_{i=1}^{N}y^{1-\epsilon}\right)^{1/(1-\epsilon)}\end{equation}$

Donde $y_{i}$ es el ingreso individual y $\mu$ es el ingreso medio

In [44]:
resultados_list = []

# Para cada trimeste
for t in [1, 2, 3, 4]:
    salario = salario_dict.get(t)
    col_ingr = f'ingr_t_t{t}'
    umbral = umbral_dict.get(t)
    
    # Agrupa por región
    grouped = data.groupby('region')
    
    for region_name, group in grouped:
        # 1. Filtra datos
        valid = group.dropna(subset=[col_ingr])
        
        if len(valid) == 0:
            continue
            
        # Extrae los vectores 
        ingresos = valid[col_ingr].values
        pesos = valid['fexp'].values
        
        # 2. Calcula indices
        gaps = (umbral - ingresos) / umbral
        gaps = np.clip(gaps, a_min=0, a_max=None)
        
        # 3. Calcula FGT
        total_poblacion = pesos.sum()
        
        # FGT0
        fgt0 = (pesos * (gaps > 0).astype(int)).sum() / total_poblacion
        
        # FGT1
        fgt1 = (pesos * (gaps ** 1)).sum() / total_poblacion
        
        # FGT2
        fgt2 = (pesos * (gaps ** 2)).sum() / total_poblacion
        
        # 4. Calcula Ingreso promedio
        ingreso_promedio = np.average(ingresos, weights=pesos)

        # 5. Desigualdad de Atkinson
        atkinson_resultados = {}
        
        if ingreso_promedio > 0:
            for epsilon in [0.25, 0.5, 0.75]:
                # La suma ponderada de la utilidad
                utility_sum = np.sum((ingresos ** (1 - epsilon)) * pesos)
                
                # promedio de esa utilidad
                utility_mean = utility_sum / total_poblacion
                
                # ingreso equivalente
                y_ede = utility_mean ** (1 / (1 - epsilon))
                
                # índice final
                atkinson_index = 1 - (y_ede / ingreso_promedio)
                atkinson_resultados[f'a{int(epsilon*100)}'] = atkinson_index
        else:
            # Si nadie gana nada, definimos desigualdad como NaN
            atkinson_resultados = {'a25': np.nan, 'a50': np.nan, 'a75': np.nan}

        # 6. Calcula mediana del ingreso
        # Ordena
        sort_idx = np.argsort(ingresos)
        ingreso_ordenado = ingresos[sort_idx]
        pesos_ordenado = pesos[sort_idx]
        cumsum_pesos = np.cumsum(pesos_ordenado)
        cutoff = total_poblacion / 2.0
        mediana = ingreso_ordenado[np.searchsorted(cumsum_pesos, cutoff)]
        
        # 7. Guarda resultados
        resultados_list.append({
            'ano': ano,
            'trimestre': t,
            'region': region_name,
            'fgt0': fgt0,
            'fgt1': fgt1,
            'fgt2': fgt2,
            'a25': atkinson_resultados['a25'],
            'a50': atkinson_resultados['a50'],
            'a75': atkinson_resultados['a75'],
            'ingreso_promedio': ingreso_promedio,
            'ingreso_mediana': mediana + 1 if mediana < 1 else mediana,
            'salario_minimo': salario,
            'kaitz_indice': salario / (mediana + 1 if mediana < 1 else mediana)         
        })

# lista a DataFrame
df_final_regional = pd.DataFrame(resultados_list)
df_final_regional

,ano,trimestre,region,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio,ingreso_mediana,salario_minimo,kaitz_indice
0,2001,1,Amazonía,0.936484,0.622432,0.457419,6.907589e-02,1.351241e-01,1.979413e-01,31.634774,24.105218,85.65,3.553173
1,2001,1,Azuay,0.910083,0.619318,0.464149,8.706845e-02,1.724424e-01,2.551168e-01,32.750891,25.305750,85.65,3.384606
2,2001,1,"Costa, Santo Domingo",1.000000,0.607768,0.408061,3.677969e-02,7.628559e-02,1.180780e-01,30.911898,28.983981,85.65,2.955081
3,2001,1,El Oro,0.987754,0.621534,0.455384,6.263713e-02,1.280139e-01,1.960997e-01,30.064656,22.334029,85.65,3.834955
4,2001,1,Guayas,0.877116,0.629990,0.497456,1.614603e-01,2.989789e-01,4.192281e-01,38.453400,19.753362,85.65,4.335971
5,2001,1,Los Ríos,0.923234,0.688471,0.560615,1.211092e-01,2.443259e-01,3.657231e-01,25.838276,12.404376,85.65,6.904821
6,2001,1,Manabí,0.872976,0.592300,0.448630,1.622353e-01,2.982927e-01,4.085655e-01,50.991823,24.758860,85.65,3.459368
7,2001,1,Pichincha,0.786717,0.492672,0.362427,1.562535e-01,2.822086e-01,3.893260e-01,58.534432,31.927988,85.65,2.682599
8,2001,1,Sierra,0.954882,0.733112,0.598299,1.161059e-01,2.175876e-01,3.070196e-01,23.474533,13.996491,85.65,6.119391
9,2001,2,Amazonía,0.976946,0.637633,0.480712,7.800758e-02,1.536579e-01,2.264383e-01,30.170239,26.295330,85.65,3.257232


### Inserta los cálculos en la base final

In [45]:
indices = pd.read_csv("indices_region.csv", encoding='latin-1')

In [46]:
import os

# 2. cheque el archivo
if not os.path.isfile('indices_region.csv'):
    # Headers si es la primera vez
    df_final_regional.to_csv('indices_region.csv', index=False, encoding='latin-1')
else:
    # SI ya existe, append
    df_final_regional.to_csv('indices_region.csv', mode='a', index=False, header=False, encoding='latin-1')